# **Two-Dimensional Arrays**

**DS112 — Computer Programming**

**Week 11:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Apply 2D array operations — insertion, deletion, searching, sorting, concatenation, splitting, and broadcasting — in simple programming problems.

A **two-dimensional (2D) array** is a table of numbers with **rows** and **columns** — like a gradebook where each row is a student and each column is an assessment. The two dimensions are called **axes**:

* **axis 0** runs **down** the rows;
* **axis 1** runs **across** the columns.

```text
                axis 1 →
               quiz  lab  midterm  final
           ┌──────────────────────────────┐
 axis 0  0 │  88    92     79       85    │  Ana
   ↓     1 │  75    80     70       72    │  Ben
         2 │  91    85     88       94    │  Carlo
         3 │  62    70     58       66    │  Diana
           └──────────────────────────────┘
                     shape = (4, 4)
```

In [ ]:
import numpy as np

names = np.array(["Ana", "Ben", "Carlo", "Diana"])
grades = np.array([
    [88, 92, 79, 85],
    [75, 80, 70, 72],
    [91, 85, 88, 94],
    [62, 70, 58, 66],
])
print(grades)
print(grades.shape, grades.ndim, grades.size)

## **Creating 2D Arrays**

In [ ]:
print(np.zeros((2, 3)))
print(np.arange(12).reshape(3, 4))     # 12 numbers arranged as 3 rows × 4 columns
print(np.eye(3, dtype=int))            # identity matrix

## **Indexing and Slicing**

Use `array[row, column]`. A `:` means "all" along that axis.

In [ ]:
print(grades[2, 3])          # Carlo's final
print(grades[1])             # Ben's row
print(grades[:, 0])          # every quiz score (column 0)
print(grades[:2, 1:3])       # first two students, lab and midterm
print(grades[-1, -1])        # last row, last column

Boolean masks work on rows, too:

In [ ]:
final_col = grades[:, 3]
print(names[final_col >= 75])            # students whose final is at least 75
print(grades[final_col >= 75])           # their full rows

## **Insertion and Deletion**

Arrays have a fixed size, so `np.insert`, `np.append`, and `np.delete` **return new arrays** — the original is unchanged.

In [ ]:
new_student = np.array([[80, 85, 77, 90]])
with_ella = np.append(grades, new_student, axis=0)              # add a row at the end
print(with_ella.shape)

bonus_col = np.array([5, 0, 3, 2])
with_bonus = np.insert(grades, 4, bonus_col, axis=1)            # add a column at position 4
print(with_bonus)

without_ben = np.delete(grades, 1, axis=0)                      # remove row 1
without_lab = np.delete(grades, 1, axis=1)                      # remove column 1
print(without_ben.shape, without_lab.shape, grades.shape)

> Because each insertion copies the whole array, avoid inserting inside a loop. Collect values in a **list** first, then convert to an array once.

## **Searching**

In [ ]:
rows, cols = np.where(grades < 65)                 # positions that satisfy a condition
print(list(zip(names[rows], cols)))

print(grades.argmax())                              # position in the flattened array
print(np.unravel_index(grades.argmax(), grades.shape))   # as (row, column)

print(np.isin(grades, [70, 85]))                    # membership test, element by element
print(np.any(grades < 60), np.all(grades > 50))

## **Sorting**

In [ ]:
print(np.sort(grades, axis=1))          # sort within each row
print(np.sort(grades, axis=0))          # sort within each column

order = np.argsort(grades[:, 3])[::-1]  # row order by final, highest first
print(names[order])
print(grades[order])                    # whole rows reordered together

`np.sort` sorts values but loses the link to other columns. **`argsort`** returns the **order of positions**, which you can use to reorder whole rows — the standard way to "sort a table by one column".

## **Concatenation and Splitting**

In [ ]:
section_a = np.array([[88, 92], [75, 80]])
section_b = np.array([[91, 85], [62, 70]])

print(np.concatenate([section_a, section_b], axis=0))   # stack rows
print(np.vstack([section_a, section_b]).shape)           # same as axis=0
print(np.hstack([section_a, section_b]))                 # side by side (axis=1)

In [ ]:
top, bottom = np.vsplit(grades, 2)               # into 2 groups of rows
left, right = np.hsplit(grades, [2])             # columns before / from index 2
print(top.shape, bottom.shape, left.shape, right.shape)

parts = np.array_split(np.arange(10), 3)         # uneven splits allowed
print(parts)

## **Broadcasting**

**Broadcasting** lets NumPy combine arrays of **different shapes** by automatically "stretching" the smaller one, without copying data.

**Rules.** Compare shapes from the **right**. Two dimensions are compatible if they are **equal** or one of them is **1**. A missing dimension counts as 1.

```text
grades           (4, 4)
weights             (4,)   → treated as (1, 4) → stretched down all 4 rows      ✓
column_means     (4, 1)    → stretched across all 4 columns                      ✓
(4, 4) and (3,)            → 4 ≠ 3, neither is 1                                 ✗ error
```

> **Analogy.** Broadcasting is like a teacher announcing *"add 5 points to the lab column"* to the whole class at once — the single instruction is applied to every row without being written out 40 times.

In [ ]:
weights = np.array([0.2, 0.3, 0.2, 0.3])           # shape (4,)
weighted = grades * weights                        # (4, 4) * (4,) → each row times the weights
print(weighted.sum(axis=1).round(2))               # final grade per student

column_means = grades.mean(axis=0)                 # shape (4,)
print((grades - column_means).round(1))            # deviation from each assessment's mean

row_max = grades.max(axis=1, keepdims=True)        # shape (4, 1)
print((grades / row_max).round(2))                 # each score relative to the student's best

try:
    grades + np.array([1, 2, 3])
except ValueError as error:
    print("ValueError:", error)

## **Summary**

| Operation | Functions |
| --------- | --------- |
| create | `np.array(nested_list)`, `reshape`, `zeros((r, c))`, `eye` |
| index / slice | `a[r, c]`, `a[:, c]`, `a[r1:r2, c1:c2]`, masks |
| insert / delete | `np.insert`, `np.append`, `np.delete` (return new arrays) |
| search | `np.where`, `argmax` + `unravel_index`, `np.isin`, `any`, `all` |
| sort | `np.sort(axis=…)`, `argsort` to reorder rows |
| combine / split | `concatenate`, `vstack`, `hstack`, `vsplit`, `hsplit`, `array_split` |
| broadcasting | shapes compatible from the right when equal or 1 |

## **References**

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

NumPy Developers. (2026). *Broadcasting*. https://numpy.org/doc/stable/user/basics.broadcasting.html

NumPy Developers. (2026). *NumPy documentation*. https://numpy.org/doc/stable/

VanderPlas, J. (2023). *Python data science handbook: Essential tools for working with data* (2nd ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/